# Importador de datos desde un PDF

## Bibliotecas necesarias

In [ ]:
# Dependencies:
#
import pandas as pd
import numpy as np
import re
import PyPDF2
import seaborn as sns
import matplotlib.pyplot as plt

#
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#!pip install PyPDF2

In [ ]:
# Opciones de visualización de cifras:
#pd.options.display.float_format = '{:,.2f}'.format #'${:,.2f}'

In [ ]:
# Opciones de visualización de dataframes:
#pd.set_option('display.max_rows', 500)
#pd.set_option('display.max_columns', 500)
#pd.set_option('display.width', 1000)

# Información en PDF

### Importación y limpieza de datos

https://www.gob.mx/cne/documentos/precios-maximos-aplicables-de-gas-lp-399897

In [ ]:
# Inicializar una lista para almacenar cada línea del PDF como un registro
datos = []

# Abrir el archivo PDF en modo binario
with open('PRECIOS_MA_XIMOS_VIGENTES_DEL_01_AL_07_DE_MARZO_DE_2026.pdf', 'rb') as archivo_pdf:
    lector_pdf = PyPDF2.PdfReader(archivo_pdf)
    # Iterar a través de cada página del PDF
    for pagina in lector_pdf.pages:
        # Extraer el texto de la página
        texto_pagina = pagina.extract_text()
        # Dividir el texto en líneas
        lineas = texto_pagina.split('\n')
                # Añadir cada línea a la lista de datos
        for linea in lineas:
            # Aquí podrías necesitar procesar cada línea si necesitas dividirla en columnas
            datos.append(linea)

#datos

In [ ]:
#

datos

In [ ]:
# Write a TXT file with data from PDF:

with open('Precios.txt', 'w') as f:
    for line in datos:
        f.write(line)
        f.write('\n')

In [ ]:
# Iteremos en los datos para extraer solo los textos de interes

datos_clean = []

for element in datos:
    if re.search( r'^\d.*\$' , element):
        datos_clean.append( element )

#datos_clean

In [ ]:
#

datos_clean

In [ ]:
# Iteremos en los datos para extraer solo los textos de interes

Texto     = []

Precio_Kg = []

Precio_Lt = []

for text in datos_clean:
    #
    match = re.search( r'^(.*?)\$([\d.]+) \$([\d.]+)', text)
    #
    Texto.append( match.group(1) )
    Precio_Kg.append( match.group(2) )
    Precio_Lt.append( match.group(3) )
#

In [ ]:
# Creamos un Dataframe

DF_GasLP = pd.DataFrame( { 'Texto': Texto,
                           'Precio_Kg': Precio_Kg,
                           'Precio_Lt': Precio_Lt
                         } )

DF_GasLP

In [ ]:
# Save into CSV:

DF_GasLP.to_csv( 'Precios.csv', encoding = 'UTF-8', index = False )

In [ ]:
# Density Plots

# Crear un KDE para la columna 'data_column'
sns.kdeplot( DF_GasLP['Precio_Kg'].astype(float) )

# Añadir títulos y etiquetas según sea necesario
plt.title('KDE del Precio Máximo por kilogramo de Gas LP')
plt.xlabel('Valores')
plt.ylabel('Densidad')

# Mostrar el gráfico
plt.show()

In [ ]:
# Density Plots

# Crear un KDE para la columna 'data_column'
sns.kdeplot( DF_GasLP['Precio_Lt'].astype(float) )

# Añadir títulos y etiquetas según sea necesario
plt.title('KDE del Precio Máximo por litro de Gas LP')
plt.xlabel('Valores')
plt.ylabel('Densidad')

# Mostrar el gráfico
plt.show()

## Para pensar

1. El código de este notebook asume una estructura fija en el PDF (mismas columnas, mismo formato de línea) para poder extraer los datos como texto y convertirlos en tabla. ¿Qué pasaría si el PDF de origen cambiara ligeramente su formato (por ejemplo, una columna nueva, o un salto de línea distinto)?

2. **Ejercicio:** en esta misma carpeta hay varios PDFs de "Precios Máximos Vigentes" de distintas fechas (febrero 2024, junio 2025, marzo 2026, mayo 2026). Corre el mismo proceso de extracción sobre uno de los PDFs más recientes en vez del que usa el notebook por defecto. ¿El código funciona sin cambios? Si falla o produce datos incorrectos, ¿en qué parte del proceso se rompe, y por qué?

3. A partir del ejercicio anterior: ¿qué harías distinto si tuvieras que automatizar esta extracción para que corriera cada semana con un PDF nuevo, sin supervisión humana?
